# 05 -- Bokeh

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer des graphiques interactifs avec `bokeh.plotting` ;
- utiliser `figure`, `ColumnDataSource` et les glyphes ;
- ajouter tooltips, hover, zoom, pan et box select ;
- combiner plusieurs graphiques avec `gridplot`, `row`, `column` ;
- styliser titres, axes, legendes ;
- creer des graphiques lies (linked brushing) ;
- exporter en HTML autonome.

## Prerequis -- ce que vous devez maitriser

- Python intermediaire et Pandas ;
- comprendre les bases d'un DataFrame.

Ce que ce notebook **ne couvrira pas** :

- les applications Bokeh Server ;
- Plotly et Altair (voir notebooks dedies).

## Plan

1. Installation et sortie
2. Premiere figure
3. ColumnDataSource
4. Glyphes (circles, lines, bars)
5. Axes et titres
6. Tooltips et hover
7. Outils interactifs
8. Layout : row, column, gridplot
9. Linked brushing
10. Couleurs et palettes
11. Legendes
12. Date axis
13. Export HTML
14. Synthese
15. Exercices

---

## 1. Installation et sortie

In [ ]:
# pip install bokeh
import bokeh
bokeh.__version__

In [ ]:
from bokeh.io import output_notebook, show
output_notebook()

`output_notebook()` affiche les figures directement dans le notebook. Pour fichier HTML standalone, utiliser `output_file('page.html')`.

---

## 2. Premiere figure

In [ ]:
from bokeh.plotting import figure
p = figure(title='Sinus', width=500, height=300)

import numpy as np
x = np.linspace(0, 10, 100)
p.line(x, np.sin(x), line_width=2)
show(p)

---

## 3. ColumnDataSource

`ColumnDataSource` est la structure centrale : elle stocke les donnees et permet les liens entre plusieurs figures.

In [ ]:
from bokeh.models import ColumnDataSource
import pandas as pd

df = pd.DataFrame({
    'x': [1, 2, 3, 4],
    'y': [2, 5, 3, 7],
    'label': ['A', 'B', 'C', 'D'],
})
source = ColumnDataSource(df)
source

In [ ]:
p = figure(width=400, height=300)
p.circle(x='x', y='y', size=15, source=source)
show(p)

---

## 4. Glyphes

In [ ]:
p = figure(width=400, height=300)
p.circle([1, 2, 3], [4, 5, 6], size=20, color='navy', alpha=0.5)
show(p)

In [ ]:
p = figure(width=400, height=300)
p.line([1, 2, 3, 4], [1, 4, 2, 6], line_width=3, color='firebrick')
show(p)

In [ ]:
# Barres verticales
p = figure(x_range=['A', 'B', 'C', 'D'], width=400, height=300)
p.vbar(x=['A', 'B', 'C', 'D'], top=[3, 7, 2, 5], width=0.7)
show(p)

In [ ]:
# Scatter
p = figure(width=400, height=300)
p.scatter([1, 2, 3], [4, 5, 6], marker='triangle', size=20)
show(p)

---

## 5. Axes et titres

In [ ]:
p = figure(title='Mon graphique', x_axis_label='X', y_axis_label='Y', width=400, height=300)
p.circle([1, 2, 3], [4, 5, 6], size=15)
p.title.text_font_size = '16pt'
p.xaxis.axis_label_text_font_style = 'bold'
show(p)

---

## 6. Tooltips et hover

In [ ]:
from bokeh.models import HoverTool

p = figure(width=400, height=300, tools='pan,wheel_zoom,reset')
p.circle(x='x', y='y', size=15, source=source)

hover = HoverTool(tooltips=[
    ('label', '@label'),
    ('coord', '(@x, @y)'),
])
p.add_tools(hover)
show(p)

---

## 7. Outils interactifs

In [ ]:
TOOLS = 'pan,box_zoom,wheel_zoom,reset,save,box_select,lasso_select,hover'
p = figure(width=400, height=300, tools=TOOLS)
p.circle([1, 2, 3, 4], [2, 5, 3, 7], size=15)
show(p)

### Outils courants

| Outil | Effet |
|---|---|
| `pan` | deplacer le graphique |
| `box_zoom` | zoomer sur une zone |
| `wheel_zoom` | zoomer avec la molette |
| `reset` | reset de la vue |
| `save` | sauvegarder en PNG |
| `hover` | info-bulle au survol |
| `box_select` | selection rectangulaire |

---

## 8. Layout : row, column, gridplot

In [ ]:
from bokeh.layouts import row, column, gridplot

p1 = figure(width=300, height=200, title='Fig 1')
p1.circle([1, 2, 3], [1, 4, 2])

p2 = figure(width=300, height=200, title='Fig 2')
p2.line([1, 2, 3], [3, 5, 4])

show(row(p1, p2))

In [ ]:
show(column(p1, p2))

In [ ]:
p3 = figure(width=300, height=200, title='Fig 3')
p3.vbar(x=[1, 2, 3], top=[2, 4, 3], width=0.5)

p4 = figure(width=300, height=200, title='Fig 4')
p4.scatter([1, 2, 3], [4, 2, 5], size=15)

show(gridplot([[p1, p2], [p3, p4]]))

---

## 9. Linked brushing

En partageant une `ColumnDataSource`, les selections se propagent entre les figures.

In [ ]:
source = ColumnDataSource(pd.DataFrame({
    'x': [1, 2, 3, 4, 5],
    'y1': [2, 5, 3, 7, 4],
    'y2': [3, 1, 4, 2, 5],
}))

p1 = figure(width=300, height=200, tools='box_select,reset')
p1.circle('x', 'y1', source=source, size=15)

p2 = figure(width=300, height=200, tools='box_select,reset')
p2.circle('x', 'y2', source=source, size=15)

show(row(p1, p2))

---

## 10. Couleurs et palettes

In [ ]:
from bokeh.palettes import Category10, Viridis256, Spectral6
print('Category10 :', Category10[10][:5])
print('Spectral6  :', Spectral6)

In [ ]:
from bokeh.transform import factor_cmap
fruits = ['pommes', 'poires', 'cerises', 'raisins']
counts = [5, 3, 4, 2]
source = ColumnDataSource(data={'fruits': fruits, 'counts': counts})
p = figure(x_range=fruits, width=400, height=300)
p.vbar(
    x='fruits', top='counts', source=source, width=0.8,
    fill_color=factor_cmap('fruits', palette=Spectral6[:4], factors=fruits),
)
show(p)

---

## 11. Legendes

In [ ]:
p = figure(width=400, height=300)
p.line([1, 2, 3], [1, 4, 2], legend_label='Serie A', color='blue')
p.line([1, 2, 3], [3, 1, 5], legend_label='Serie B', color='red')
p.legend.location = 'top_left'
p.legend.click_policy = 'hide'  # cliquer pour masquer
show(p)

---

## 12. Date axis

In [ ]:
dates = pd.date_range('2024-01-01', periods=30)
values = np.cumsum(np.random.default_rng(0).standard_normal(30))
p = figure(x_axis_type='datetime', width=500, height=300)
p.line(dates, values, line_width=2)
show(p)

---

## 13. Export HTML

In [ ]:
from bokeh.io import output_file, save
import tempfile, pathlib
out = pathlib.Path(tempfile.gettempdir()) / 'bokeh_plot.html'
# output_file(str(out))
# save(p)
print('HTML genere dans :', out)

---

## Synthese

| Element | Bokeh |
|---|---|
| Figure | `figure(title=..., tools=...)` |
| Source | `ColumnDataSource(df)` |
| Ligne | `p.line(x, y, ...)` |
| Point | `p.circle(x, y, size=...)` |
| Barres | `p.vbar(x, top, width=...)` |
| Tooltips | `HoverTool(tooltips=[...])` |
| Layout | `row`, `column`, `gridplot` |
| Legende | `legend_label='...'` + `p.legend.*` |
| Export | `output_file(...)` + `save(p)` |

### Regles a retenir

- **Une `figure` par graphique**, on y ajoute les glyphes.
- **ColumnDataSource** pour donnees partagees et linked brushing.
- **Tooltips via `HoverTool`** pour l'interactivite.
- **`show` dans notebook** apres `output_notebook()`.

---

## Exercices

### Exercice 1 (facile) -- Line chart

Affichez `y = x**2` pour x dans [0, 10] avec 100 points, avec titre et axes nommes.

In [ ]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()

# TODO

<details>
<summary>Voir la correction</summary>

```python
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()

x = np.linspace(0, 10, 100)
p = figure(title='Parabole', x_axis_label='x', y_axis_label='y', width=500, height=300)
p.line(x, x**2, line_width=2)
show(p)
```

</details>

### Exercice 2 (facile) -- Barres

Faites un bar chart des ventes `[A=10, B=15, C=7, D=12]` avec couleurs differentes.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from bokeh.plotting import figure, show
from bokeh.palettes import Spectral4

cats = ['A', 'B', 'C', 'D']
vals = [10, 15, 7, 12]
p = figure(x_range=cats, width=400, height=300, title='Ventes')
p.vbar(x=cats, top=vals, width=0.7, color=list(Spectral4))
show(p)
```

</details>

### Exercice 3 (moyen) -- Scatter avec hover

Creez un DataFrame avec `x`, `y`, `name` (3 points), affichez un scatter avec tooltip qui montre le nom.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource

df = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 2, 5], 'name': ['Alice', 'Bob', 'Eve']})
source = ColumnDataSource(df)
p = figure(width=400, height=300)
p.circle('x', 'y', size=20, source=source)
p.add_tools(HoverTool(tooltips=[('name', '@name'), ('pos', '(@x, @y)')]))
show(p)
```

</details>

### Exercice 4 (moyen) -- Gridplot

Construisez un gridplot 2x2 avec : ligne, scatter, barres, courbe.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
import numpy as np

x = np.linspace(0, 10, 50)
p1 = figure(width=250, height=200, title='Ligne')
p1.line(x, np.sin(x))
p2 = figure(width=250, height=200, title='Scatter')
p2.circle(x, np.cos(x), size=6)
p3 = figure(width=250, height=200, title='Barres', x_range=['A', 'B', 'C'])
p3.vbar(x=['A', 'B', 'C'], top=[3, 5, 2], width=0.6)
p4 = figure(width=250, height=200, title='Courbe')
p4.line(x, x**2 / 10)
show(gridplot([[p1, p2], [p3, p4]]))
```

</details>

### Exercice 5 (difficile) -- Linked brushing

Avec une `ColumnDataSource` partagee, creez deux graphiques lies (x, y1) et (x, y2) ou la selection rectangle se propage.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from bokeh.layouts import row

src = ColumnDataSource(pd.DataFrame({
    'x': list(range(10)),
    'y1': [2, 5, 3, 7, 4, 8, 6, 9, 5, 7],
    'y2': [3, 1, 4, 2, 5, 3, 6, 4, 7, 5],
}))
TOOLS = 'box_select,reset'
p1 = figure(width=300, height=250, tools=TOOLS)
p1.circle('x', 'y1', source=src, size=15)
p2 = figure(width=300, height=250, tools=TOOLS)
p2.circle('x', 'y2', source=src, size=15)
show(row(p1, p2))
```

</details>

---

## Ressources externes

- [Documentation Bokeh](https://docs.bokeh.org/)
- [Gallery](https://docs.bokeh.org/en/latest/docs/gallery.html)
- [User guide](https://docs.bokeh.org/en/latest/docs/user_guide.html)